## Set Install Libraries and Set Environment

In [ ]:
!pip install pandas matplotlib seaborn wordcloud textblob pypdf tqdm scikit-learn --quiet
!pip install -U openai --quiet

import pandas as pd, json, re, textwrap, matplotlib.pyplot as plt, seaborn as sns, os, glob, logging
from pypdf import PdfReader
from google.colab import files, userdata
from wordcloud import WordCloud
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

sns.set_theme(style="whitegrid")
logging.getLogger("pypdf").setLevel(logging.ERROR) ### hide harmless pdf formatting warnings

OUTPUT_DIR = "outputs" ### all csv files and charts are saved here
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Set up API

In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") ### reads key saved in Colab Secrets (key icon on left sidebar) - turn on notebook access when asked

## Choose Model and Connect to API

In [ ]:
client = OpenAI(max_retries=5) ### local python object to make API calls - retries if rate limited
MODEL = "gpt-6-luna" ### model used in all cells below - change here to try another model
print("Connected to QualLLM!")
print("Model:", MODEL)

## Make a Request to Model

In [ ]:
response = client.responses.create(
    model=MODEL,
    input=[{
        "role": "user",
        "content": [{"type": "input_text", "text": "Explain how you analyze qualitative data."}]
    }]
)
print(response.output_text)

## Load Interview Data

In [ ]:
### option 1: use 10 workshop interviews from GitHub repo
if not os.path.exists("llms-for-qual"):
    !git clone -q https://github.com/namigabbasov/llms-for-qual.git
paths = sorted(glob.glob("llms-for-qual/data/*.pdf"))

### option 2: upload your own PDF or TXT transcripts - uncomment 2 lines below
# uploaded = files.upload()
# paths = sorted(uploaded.keys())

print(f"Found {len(paths)} interview files")

In [ ]:
### read and clean text of each interview
docs = {}
for path in paths:
    name = os.path.splitext(os.path.basename(path))[0]
    if path.lower().endswith(".pdf"):
        reader = PdfReader(path)
        text = " ".join(p.extract_text() or "" for p in reader.pages)
    else:
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
    docs[name] = re.sub(r"\s+", " ", text).strip()

print(f"Loaded {len(docs)} documents:")
for d in docs:
    print(" •", d)

In [ ]:
### list to hold file IDs for all PDFs
file_ids = []

### Register each file with OpenAI API
for path in paths:
    with open(path, "rb") as f:
        file_obj = client.files.create(file=f, purpose="user_data")
    file_ids.append(file_obj.id)
    print(f"Uploaded {os.path.basename(path)} → File ID: {file_obj.id}")

print("\nAll files uploaded to OpenAI successfully!")

In [ ]:
### ask model about uploaded PDFs - feel free to change question below
content = [{"type": "input_file", "file_id": fid} for fid in file_ids]
content.append({"type": "input_text", "text": "Summarize the main ideas raised across these interviews."})

response = client.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": content}]
)
print(response.output_text)

## Exploratory Data Analysis

In [ ]:
df = pd.DataFrame(list(docs.items()), columns=["filename", "text"])
df["word_count"] = df["text"].apply(lambda x: len(x.split()))
df["char_count"] = df["text"].apply(len)
df["avg_word_length"] = df["text"].apply(lambda x: sum(len(w) for w in x.split()) / max(len(x.split()), 1))

print("Basic stats per document:")
display(df[["filename", "word_count", "char_count", "avg_word_length"]])
df[["filename", "word_count", "char_count", "avg_word_length"]].to_csv(f"{OUTPUT_DIR}/document_stats.csv", index=False)

In [ ]:
plt.figure(figsize=(8,4))
sns.barplot(y="filename", x="word_count", data=df, hue="filename", palette="viridis", legend=False)
plt.title("Document Lengths (Word Count)")
plt.xlabel("Words per Document"); plt.ylabel("Interview File")
plt.savefig(f"{OUTPUT_DIR}/document_lengths.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
corpus = " ".join(df["text"])
wc = WordCloud(width=1200, height=600, background_color="white", colormap="viridis").generate(corpus)
plt.figure(figsize=(12,6))
plt.imshow(wc, interpolation='bilinear')
plt.axis("off")
plt.title("Word Cloud: Combined Interviews", fontsize=14)
plt.savefig(f"{OUTPUT_DIR}/word_cloud.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
vectorizer = CountVectorizer(stop_words="english", max_features=1000)
X = vectorizer.fit_transform(df["text"])
word_freq = pd.DataFrame({
    "word": vectorizer.get_feature_names_out(),
    "count": X.toarray().sum(axis=0)
}).sort_values("count", ascending=False)

plt.figure(figsize=(8,4))
sns.barplot(y="word", x="count", data=word_freq.head(20))
plt.title("Top 20 Most Frequent Words (All Interviews)")
plt.xlabel("Frequency"); plt.ylabel("Word")
plt.savefig(f"{OUTPUT_DIR}/top_words.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df["sentiment"] = df["text"].apply(lambda x: TextBlob(x).sentiment.polarity)
plt.figure(figsize=(8,4))
sns.barplot(y="filename", x="sentiment", data=df)
plt.title("Average Sentiment by Interview")
plt.xlabel("Sentiment Polarity (-1 to 1)"); plt.ylabel("Document")
plt.savefig(f"{OUTPUT_DIR}/sentiment_by_interview.png", dpi=150, bbox_inches="tight")
plt.show()

sns.histplot(df["sentiment"], bins=20, kde=True, color="purple")
plt.title("Sentiment Distribution Across All Interviews")
plt.xlabel("Polarity"); plt.ylabel("Frequency")
plt.show()

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", max_features=1500)
X_tfidf = tfidf.fit_transform(df["text"])
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out(), index=df["filename"])

top_terms = pd.DataFrame(tfidf_df.mean().sort_values(ascending=False).head(20))
top_terms.columns = ["mean_tfidf"]
top_terms.plot(kind="barh", figsize=(8,5), color="teal")
plt.gca().invert_yaxis()
plt.title("Top 20 Distinctive Words (Corpus-Wide TF–IDF)")
plt.xlabel("Average TF–IDF Score")
plt.savefig(f"{OUTPUT_DIR}/top_tfidf_words.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(X_tfidf.toarray())
plt.figure(figsize=(7,6))
plt.scatter(coords[:,0], coords[:,1], alpha=0.7, color="teal")
for i, fn in enumerate(df["filename"]):
    plt.text(coords[i,0]+0.02, coords[i,1], fn.split('.')[0], fontsize=9)
plt.title("Semantic Map of Interviews (PCA of TF–IDF)") ### rough similarity in terms of words not true semantic
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.savefig(f"{OUTPUT_DIR}/interview_map_pca.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
topics = []
for text in tqdm(df["text"], desc="Generating GPT topic summaries"):
    prompt = f"List 3-5 concise topic keywords describing main ideas in this interview:\n\n{text}"
    resp = client.responses.create(
        model=MODEL,
        input=[{"role": "user", "content": [{"type": "input_text", "text": prompt}]}]
    )
    topics.append(resp.output_text.strip())

df["gpt_topics"] = topics
print("GPT Topic Labels Added")
display(df[["filename", "gpt_topics"]])
df[["filename", "gpt_topics"]].to_csv(f"{OUTPUT_DIR}/gpt_topics.csv", index=False)

## Deductive Thematic Analysis

###*RQ: How do university instructors adapt their data-teaching practices to meet diverse student needs and evolving technological and institutional contexts?*

In [ ]:
### define themes
codes = [
    "Student-Centered Adjustments",
    "Tool and Technology Shifts",
    "Curriculum and Pedagogical Redesign",
    "Professional Learning and Skill Growth",
    "Collaborative and Institutional Support",
    "Relevance and Engagement Strategies"
]

### split each interview on speaker labels - I = interviewer, UCSB01, UCSB02 ... = instructor
turn_rows = []
for name, text in docs.items():
    parts = re.split(r"(\bI\s*:|\b(?i:UCSB)\s*\d*\s*:)", text)
    speakers, turns = parts[1::2], parts[2::2]

    ### keep only instructor responses
    kept = [t.strip() for s, t in zip(speakers, turns) if s.upper().startswith("UCSB") and len(t.strip()) > 80]
    for j, t in enumerate(kept):
        turn_rows.append({"interview": name, "chunk_id": j + 1, "text": t})
    print(f"{name}: split into {len(turns)} turns, keeping {len(kept)} instructor responses")

turns_df = pd.DataFrame(turn_rows)
print(f"\n{len(turns_df)} instructor responses across {turns_df['interview'].nunique()} interviews")

### preview few chunks
for _, r in turns_df.head(3).iterrows():
    print(f"\n--- {r['interview']} RESPONSE {r['chunk_id']} ---\n{r['text'][:250]}...\n")

In [ ]:
### coding function for one response
def code_deductive(r):
    prompt = f"""
    You are a qualitative coding assistant. Read the passage and decide whether each of {len(codes)} themes appears.
    Return ONLY valid JSON with each key = theme and value = 1 (present) or 0 (absent).
    Themes: {', '.join(codes)}
    Passage: \"\"\"{textwrap.shorten(r['text'], width=6000)}\"\"\"
    """
    resp = client.responses.create(
        model=MODEL,
        input=[{"role": "user", "content": [{"type": "input_text", "text": prompt}]}],
        text={"format": {"type": "json_object"}}  ### forces JSON output
    )
    raw = resp.output_text.strip()
    try:
        result = json.loads(raw)
    except Exception:
        try:
            start, end = raw.index("{"), raw.rindex("}") + 1
            result = json.loads(raw[start:end])
        except Exception:
            result = {c: None for c in codes}
    result["interview"] = r["interview"]
    result["chunk_id"] = r["chunk_id"]
    result["chunk_preview"] = r["text"][:220] + "..."
    return result

### run coding loop - 8 requests at a time to save time
records = turns_df.to_dict("records")
with ThreadPoolExecutor(max_workers=8) as pool:
    rows = list(tqdm(pool.map(code_deductive, records), total=len(records), desc="Coding instructor responses"))

In [ ]:
### create DataFrame, save, and preview
deductive_df = pd.DataFrame(rows)
deductive_df = deductive_df.reindex(columns=["interview", "chunk_id"] + codes + ["chunk_preview"])
deductive_df[codes] = deductive_df[codes].apply(pd.to_numeric, errors="coerce")
deductive_df.to_csv(f"{OUTPUT_DIR}/deductive_coding_matrix.csv", index=False)
print("saved coding matrix as deductive_coding_matrix.csv")
print("\nPreview:")
display(deductive_df.head())

### theme counts per interview
deductive_by_interview = deductive_df.groupby("interview")[codes].sum().astype(int)
deductive_by_interview.to_csv(f"{OUTPUT_DIR}/deductive_themes_by_interview.csv")
display(deductive_by_interview)

### summary counts and bar chart
theme_counts = deductive_df[codes].sum().sort_values(ascending=False)
print("\nTheme counts:\n", theme_counts)
plt.figure(figsize=(8,4))
theme_counts.plot(kind="barh", color="teal")
plt.gca().invert_yaxis()
plt.title("Frequency of Themes in Instructor Responses (Deductive)")
plt.xlabel("Number of Chunks Coded")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/deductive_theme_counts.png", dpi=150, bbox_inches="tight")
plt.show()

### heatmap of themes by interview
plt.figure(figsize=(10,5))
sns.heatmap(deductive_by_interview.T, annot=True, fmt="d", cmap="Blues", cbar_kws={"label": "Chunks Coded"})
plt.title("Themes by Interview (Deductive)")
plt.xlabel("Interview"); plt.ylabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/deductive_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Inductive Thematic Analysis

In [ ]:
### first pass: derive candidate themes from all transcripts
corpus = "\n\n".join(f"[{name}] {text}" for name, text in docs.items())
overview_prompt = f"""
You are QualLLM, a qualitative researcher.
Read following transcripts and list 5-6 inductive themes that best capture recurring ideas across interviews.
Give each theme a short label of 2-6 words.
Return ONLY valid JSON in this format:
{{"themes":["theme1","theme2","theme3",...]}}
Transcripts:
\"\"\"{corpus}\"\"\"
"""

resp = client.responses.create(
    model=MODEL,
    input=[{"role":"user","content":[{"type":"input_text","text":overview_prompt}]}],
    text={"format": {"type": "json_object"}}  ### forces JSON output
)

raw = resp.output_text.strip()
try:
    raw = re.sub(r"```(?:json)?", "", raw)
    raw = re.sub(r"```", "", raw)
    start, end = raw.find("{"), raw.rfind("}") + 1
    json_text = raw[start:end]
    theme_data = json.loads(json_text)
    themes = [t.strip() for t in theme_data.get("themes", [])]
except Exception:
    print("could not parse themes automatically. check output below:")
    print(raw)
    themes = []

print(f"derived {len(themes)} emergent themes:\n", themes)
with open(f"{OUTPUT_DIR}/inductive_themes.json", "w") as f:
    json.dump({"themes": themes}, f, indent=2)

In [ ]:
### second pass: code each chunk - 0/1 per theme - feel free to change this prompt below
def code_inductive(r):
    prompt = f"""
    You are QualLLM, a qualitative coding assistant.
    Determine whether each of the following themes appears in this passage.
    Return ONLY valid JSON with each theme as key and 1 (present) or 0 (absent) as value.

    Themes: {', '.join(themes)}

    Passage:
    \"\"\"{textwrap.shorten(r['text'], width=6000)}\"\"\"
    """
    resp = client.responses.create(
        model=MODEL,
        input=[{"role":"user","content":[{"type":"input_text","text":prompt}]}],
        text={"format": {"type": "json_object"}}  ### forces JSON output
    )

    raw = resp.output_text.strip()
    try:
        raw = re.sub(r"```(?:json)?", "", raw)
        raw = re.sub(r"```", "", raw)
        start, end = raw.find("{"), raw.rfind("}") + 1
        json_text = raw[start:end]
        result = json.loads(json_text)
    except Exception:
        result = {t: None for t in themes}

    result["interview"] = r["interview"]
    result["chunk_id"] = r["chunk_id"]
    result["chunk_preview"] = r["text"][:220] + "..."
    return result

### run coding loop - 8 requests at a time to save time
records = turns_df.to_dict("records")
with ThreadPoolExecutor(max_workers=8) as pool:
    rows = list(tqdm(pool.map(code_inductive, records), total=len(records), desc="coding chunks inductively"))

In [ ]:
### create DataFrame, save, and preview
inductive_df = pd.DataFrame(rows)
ordered_cols = ["interview", "chunk_id"] + themes + ["chunk_preview"]
inductive_df = inductive_df.reindex(columns=ordered_cols)
inductive_df[themes] = inductive_df[themes].apply(pd.to_numeric, errors="coerce")
inductive_df.to_csv(f"{OUTPUT_DIR}/inductive_coding_matrix.csv", index=False)
print("Saved inductive coding matrix as inductive_coding_matrix.csv")
display(inductive_df.head())

### theme counts per interview
inductive_by_interview = inductive_df.groupby("interview")[themes].sum().astype(int)
inductive_by_interview.to_csv(f"{OUTPUT_DIR}/inductive_themes_by_interview.csv")
display(inductive_by_interview)

### summary counts and bar chart
theme_counts = inductive_df[themes].sum().sort_values(ascending=False)
print("\nTheme counts:\n", theme_counts)

plt.figure(figsize=(8,4))
theme_counts.plot(kind="barh", color="teal")
plt.gca().invert_yaxis()
plt.title("Frequency of Emergent Themes Across Chunks (Inductive)")
plt.xlabel("Number of Chunks Coded 1")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/inductive_theme_counts.png", dpi=150, bbox_inches="tight")
plt.show()

### heatmap of themes by interview
plt.figure(figsize=(10,5))
sns.heatmap(inductive_by_interview.T, annot=True, fmt="d", cmap="Blues", cbar_kws={"label": "Chunks Coded"})
plt.title("Themes by Interview (Inductive)")
plt.xlabel("Interview"); plt.ylabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/inductive_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Download Results

In [ ]:
### zip outputs folder and download
!zip -rq outputs.zip outputs
files.download("outputs.zip")